In [4]:
!pip uninstall -y torchao
!pip install -q -U transformers peft accelerate bitsandbytes

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 20.8 MB/s eta 0:00:00


In [5]:
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

# Hugging Face SFT adapter repository
SFT_MODEL = "ksumanthreddy96/studymate-ctp-sft"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL)

# Load model with its compatible base model and adapter configuration
model = AutoPeftModelForCausalLM.from_pretrained(
    SFT_MODEL,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()

print("SFT model loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 35.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

SFT model loaded successfully!


In [6]:
def ask_ctp_question(question, max_new_tokens=150):
    prompt = f"""You are a Construction Technology and Practices (CTP) assistant.
Answer the following question clearly and accurately.

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Return only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [7]:
questions = [
    "What is fly ash?",
    "What is the effect of unburnt carbon in fly ash?",
    "Explain the difference between Class F and Class C fly ash."
]

for question in questions:
    print("Question:", question)
    print("Answer:", ask_ctp_question(question))
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is fly ash?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Fly ash is an aggregate of crushed stone or gravel, which has been ground to a fine particle size by mechanical means.
------------------------------------------------------------
Question: What is the effect of unburnt carbon in fly ash?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: The effect of unburned carbon on fly ash can be summarized as follows:

1. Increase in strength
2. Decrease in ductility
3. Reduction in compressive strength

These effects occur due to the presence of unburned carbon within the fly ash, which increases its density and reduces its bulk modulus. This results in an increase in the tensile strength but decreases in the bending resistance.
------------------------------------------------------------
Question: Explain the difference between Class F and Class C fly ash.
Answer: The main difference between Class F and Class C fly ash is that Class F has more than 10% of fine particles while Class C has less than 5%.
------------------------------------------------------------


In [9]:
question = input("Enter your CTP question: ")
answer = ask_ctp_question(question)

print("\nAnswer:")
print(answer)

Enter your CTP question: what are piles?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
Piles are long, hollow or rectangular shaped concrete blocks used to support walls, foundations, floors, etc.
